In [3]:
# Nomes das colunas de interesse nos dataframes
COL_ID_TRECHO = 'Identificador do processo de viagem' 
COL_ID_TRECHO_SPACED = 'Identificador do processo de viagem ' 
COL_ORIGEM = 'Origem - Cidade'
COL_DESTINO = 'Destino - Cidade'

from viagens_dowload import Viagens # Importa a classe Viagens do módulo viagens_dowload
ano = 2023
vi = Viagens(ano)

In [ ]:
# faz download dos dados de viagens a serviço e carrega os dataframes
# viagem_df, trecho_df, passagem_df = vi.pegarViagens()

🔄 Fazendo download de: https://portaldatransparencia.gov.br/download-de-dados/viagens/2025
➡️ Redirecionado para o CSV: https://dadosabertos-download.cgu.gov.br/PortalDaTransparencia/saida/viagens/2025_20251005_Viagens.zip
✅ CSV salvo em: dadosViagens/viagens_2025.zip
✅ Arquivos extraídos para: d:\projetoExtensao\viagensAServico\dadosViagens\dados_viagens2025
📁 Conteúdo extraído:
 - 2025_Pagamento.csv
 - 2025_Passagem.csv
 - 2025_Trecho.csv
 - 2025_Viagem.csv
✅ Arquivo ZIP removido: dadosViagens/viagens_2025.zip
✅ Arquivo de pagamentos removido: dadosViagens/dados_viagens2025/2025_Pagamento.csv
✅ 2025_Passagem.csv carregado com 250511 linhas (sem cabeçalho).
✅ 2025_Trecho.csv carregado com 1050669 linhas (sem cabeçalho).
✅ 2025_Viagem.csv carregado com 489898 linhas (sem cabeçalho).


In [4]:
#se já tiver os csvs baixados, só carrega os dataframes
viagem_df, trecho_df, passagem_df = vi.carregar_csvs()

✅ 2023_Passagem.csv carregado com 390728 linhas (sem cabeçalho).
✅ 2023_Trecho.csv carregado com 1817649 linhas (sem cabeçalho).
✅ 2023_Viagem.csv carregado com 832499 linhas (sem cabeçalho).


In [25]:
# --- CÉLULA 2 (REVISADA E OTIMIZADA): Filtragem, Limpeza e Criação da Chave ---

# --- 0. FILTRO INICIAL CRÍTICO: Situação = "Realizada" ---
# A coluna 'Situação' está no passagem_df.
VIAGEM_STATUS_FILTRO = "Realizada"

# Filtra o DataFrame de Passagens
linhas_antes = len(passagem_df)
passagem_df = passagem_df[passagem_df['Situação'] == VIAGEM_STATUS_FILTRO].copy()
linhas_depois = len(passagem_df)

print(f"✅ Filtro inicial aplicado: Mantidas {linhas_depois} viagens '{VIAGEM_STATUS_FILTRO}'.")
print(f"   ({linhas_antes - linhas_depois} viagens não realizadas foram descartadas).")


# 1. Limpeza do trecho_df
# Renomeia a coluna com erro de espaço para garantir o merge
if COL_ID_TRECHO_SPACED in trecho_df.columns:
    trecho_df = trecho_df.rename(columns={COL_ID_TRECHO_SPACED: COL_ID_TRECHO})
    print(f"✅ Coluna '{COL_ID_TRECHO_SPACED}' renomeada para '{COL_ID_TRECHO}'.")

# 2. Garante tipos de dados e limpa valores
for df in [passagem_df, trecho_df, viagem_df]:
    if COL_ID_TRECHO in df.columns:
        df[COL_ID_TRECHO] = df[COL_ID_TRECHO].astype(str).str.strip()

if COL_ORIGEM in trecho_df.columns and COL_DESTINO in trecho_df.columns:
    trecho_df[COL_ORIGEM] = trecho_df[COL_ORIGEM].astype(str).str.strip()
    trecho_df[COL_DESTINO] = trecho_df[COL_DESTINO].astype(str).str.strip()
    print("✅ Normalização das chaves e cidades concluída.")
else:
    print(f"❌ Erro: Colunas de origem/destino ({COL_ORIGEM}/{COL_DESTINO}) não encontradas no trecho_df.")


# 3. CRIAÇÃO DA CHAVE TEMPORÁRIA
trecho_df['Cod Trecho Temp'] = trecho_df[COL_ORIGEM] + '_' + trecho_df[COL_DESTINO]
trecho_df['Cod Trecho Temp'] = trecho_df['Cod Trecho Temp'].str.strip()

print("✅ Chave 'Cod Trecho Temp' criada.")

✅ Filtro inicial aplicado: Mantidas 815707 viagens 'Realizada'.
   (16792 viagens não realizadas foram descartadas).
✅ Normalização das chaves e cidades concluída.
✅ Chave 'Cod Trecho Temp' criada.


# calculando distancias

In [ ]:
# # Inicializa o objeto de cálculo de distância
# distancia_calc = Distancia() 

# def calcular_distancia_para_trecho(origem, destino):
#     """Função wrapper para Distancia.calcular_distancia."""
#     if pd.isna(origem) or pd.isna(destino) or origem.lower() in ('nan', ''):
#         return np.nan

#     try:
#         resultado = distancia_calc.calcular_distancia(origem, destino)
#         return resultado['distancia_km']
#     except Exception as e:
#         # Silencia a impressão de erro para evitar poluição no notebook
#         return np.nan

# # 1. Identificar trechos únicos para cálculo
# # Cria o Cod Trecho Temp em trecho_df
# trecho_df['Cod Trecho Temp'] = trecho_df[COL_ORIGEM] + '_' + trecho_df[COL_DESTINO]
# trechos_unicos_df = trecho_df.drop_duplicates(subset=['Cod Trecho Temp']).copy()

# print(f"\n🔄 Iniciando o cálculo automático de distâncias para {len(trechos_unicos_df)} trechos únicos...")
# start_time = time.time()

# # 2. Aplica a função de cálculo SOMENTE nos trechos únicos
# trechos_unicos_df['Distância (GCD)'] = trechos_unicos_df.apply(
#     lambda row: calcular_distancia_para_trecho(row[COL_ORIGEM], row[COL_DESTINO]), 
#     axis=1
# )

# tempo_gasto = time.time() - start_time
# print(f"✅ Cálculo concluído em {tempo_gasto:.2f} segundos.")

# # 3. Prepara a tabela de merge
# df_trechos_calculados = trechos_unicos_df[['Cod Trecho Temp', 'Distância (GCD)']].copy()


🔄 Iniciando o cálculo automático de distâncias para 100214 trechos únicos...
✅ Cálculo concluído em 5.71 segundos.


In [26]:
# --- CÉLULA 3: GEOCÓDIGO E CÁLCULO VETORIZADO ---
from distanciaCidades2 import Distancia
import pandas as pd
import numpy as np

# 1. Preparação da Classe e Lista de Cidades
# A classe Distancia agora faz todo o trabalho de gerenciar o cache e a API
geocoder = Distancia() 

# Concatena todas as cidades de origem e destino para obter a lista única
cidades_origem = trecho_df[COL_ORIGEM].unique()
cidades_destino = trecho_df[COL_DESTINO].unique()
cidades_trecho_list = np.unique(np.concatenate((cidades_origem, cidades_destino))).tolist()


# 2. Obter Mapa de Coordenadas Híbrido (Gerenciamento de API e Cache)
# A classe retorna o DataFrame mais completo de Cidades, Latitudes e Longitudes
df_coordenadas_mapa = geocoder.obter_mapa_coordenadas_hibridas(cidades_trecho_list)


# 3. Merge das Coordenadas no DataFrame de Trechos
# Merge da Origem
trecho_df = pd.merge(
    trecho_df,
    df_coordenadas_mapa.rename(columns={'Latitude': 'Lat_Origem', 'Longitude': 'Lon_Origem', 'Cidade': COL_ORIGEM}),
    on=COL_ORIGEM,
    how='left'
)
# Merge do Destino
trecho_df = pd.merge(
    trecho_df,
    df_coordenadas_mapa.rename(columns={'Latitude': 'Lat_Destino', 'Longitude': 'Lon_Destino', 'Cidade': COL_DESTINO}),
    on=COL_DESTINO,
    how='left'
)


# 4. Cálculo Vetorizado da Distância (GCD)
# Usa o método estático da classe Distancia para calcular em lote.
trecho_df['Distância (GCD)'] = Distancia.calcular_haversine(
    trecho_df['Lat_Origem'], 
    trecho_df['Lon_Origem'], 
    trecho_df['Lat_Destino'], 
    trecho_df['Lon_Destino']
)

# 5. Limpeza e Preparação para o Merge Final (Célula 4)
trecho_df = trecho_df.drop(columns=['Lat_Origem', 'Lon_Origem', 'Lat_Destino', 'Lon_Destino'], errors='ignore')

# df_trechos_calculados é a tabela de lookup para o merge final (necessário para a Célula 4)
df_trechos_calculados = trecho_df[['Cod Trecho Temp', 'Distância (GCD)']].copy()

print(f"\n✅ CÁLCULO DE DISTÂNCIA VETORIZADO CONCLUÍDO.")
print(f"Linhas com distância calculada: {df_trechos_calculados['Distância (GCD)'].notna().sum()}")

   -> Cache API carregado com sucesso: 6987 entradas.
✅ Gerenciador de Coordenadas inicializado. Mapa possui 6987 cidades.

🔄 Geocoding API: 23 cidades faltantes encontradas. Iniciando com Rate Limit (1.1s de espera).
   -> Processando: 0/23. Cidade atual: AnaheimCA - Califórnia
   -> Processando: 10/23. Cidade atual: Itajuipi
   -> Processando: 20/23. Cidade atual: Uádi Natrum
   -> Cache API carregado com sucesso: 6987 entradas.
✅ Cache FINAL sobrescrito. 6987 cidades totais. Tempo final: 42.89s.

✅ CÁLCULO DE DISTÂNCIA VETORIZADO CONCLUÍDO.
Linhas com distância calculada: 1817183


# Merger

In [63]:
# --- CÉLULA 4 (CORRIGIDA E OTIMIZADA PARA MEMÓRIA) ---

# 1. GARANTIR UNICIDADE NA TABELA DE LOOKUP (CRÍTICO)
# Se houver duas distâncias para o mesmo par de cidades, usamos a média para garantir a unicidade do índice.
df_trechos_calculados = df_trechos_calculados.groupby('Cod Trecho Temp')['Distância (GCD)'].mean().reset_index()


# 2. OTIMIZAÇÃO DE MEMÓRIA: Usar .map() em vez de pd.merge()

# Cria a tabela de lookup (Série) a partir do DataFrame menor:
# Indexa pelo 'Cod Trecho Temp' e seleciona a coluna 'Distância (GCD)'
distance_lookup = df_trechos_calculados.set_index('Cod Trecho Temp')['Distância (GCD)']

# Aplica a distância diretamente ao DataFrame grande (trecho_df).
trecho_df['Distância (GCD)'] = trecho_df['Cod Trecho Temp'].map(distance_lookup)
trecho_completo_df = trecho_df # Usa o trecho_df enriquecido como a base completa

# 3. Remove a coluna temporária *após* o lookup
trecho_completo_df = trecho_completo_df.drop(columns=['Cod Trecho Temp'])
print("✅ Distâncias anexadas via .map() (Otimização de Memória).")


# 4. Merge final: Passagem + Trechos + Viagem (Mantendo o restante do seu fluxo)
df_master = pd.merge(
    passagem_df,
    trecho_completo_df,
    on=COL_ID_TRECHO,
    how='left'
)

df_master = pd.merge(
    df_master,
    viagem_df.drop(columns=['Número da Proposta (PCDP)', 'Meio de transporte'], errors='ignore'),
    on=COL_ID_TRECHO,
    how='left',
    suffixes=('_PASSAGEM', '_VIAGEM')
)

# 5. Limpeza de colunas redundantes e renomeação
colunas_redundantes = [
    'Número da Proposta (PCDP)_y', 
    'País - Origem ida', 'UF - Origem ida', 'Cidade - Origem ida',
    'País - Destino ida', 'UF - Destino ida', 'Cidade - Destino ida',
    'País - Origem volta', 'UF - Origem volta', 'Cidade - Origem volta',
    'Pais - Destino volta', 'UF - Destino volta', 'Cidade - Destino volta'
]
df_master = df_master.drop(columns=colunas_redundantes, errors='ignore') 
df_master = df_master.rename(columns={'Número da Proposta (PCDP)_x': 'Número da Proposta (PCDP)'})

print(f"\n✅ DataFrame Mestre Final ('df_master') criado com {len(df_master)} linhas.")

✅ Distâncias anexadas via .map() (Otimização de Memória).


MemoryError: Unable to allocate 638. MiB for an array with shape (35, 2391059) and data type object

In [ ]:
df_master.head()

,Identificador do processo de viagem,Número da Proposta (PCDP),Situação,Viagem Urgente,Justificativa Urgência Viagem,Código do órgão superior,Nome do órgão superior,Código órgão solicitante,Nome órgão solicitante,CPF viajante,...,Destino - UF,Destino - Cidade,Meio de transporte,Número Diárias,Missao?,Distância (GCD),Valor da passagem,Taxa de serviço,Data da emissão/compra,Hora da emissão/compra
0,0000000000017821923,000001/23-1C,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26352,Fundação Universidade Federal do ABC,***.875.238-**,...,NaN,Loughborough,Aéreo,"0,00",Sim,9552.884444,NaN,NaN,NaN,NaN
1,0000000000017821923,000001/23-1C,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26352,Fundação Universidade Federal do ABC,***.875.238-**,...,São Paulo,São Paulo,Aéreo,"0,00",Não,9552.884444,NaN,NaN,NaN,NaN
2,0000000000018159396,000001/23,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26271,Fundação Universidade de Brasília,***.000.000-**,...,Goiás,Pirenópolis,Rodoviário,"0,00",Sim,989.393031,"3940,94","6,31",11/08/2022,20:26
3,0000000000018159396,000001/23,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26271,Fundação Universidade de Brasília,***.000.000-**,...,Goiás,Pirenópolis,Rodoviário,"0,00",Sim,989.393031,"3940,94","6,31",11/08/2022,20:26
4,0000000000018159396,000001/23,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26271,Fundação Universidade de Brasília,***.000.000-**,...,Rio de Janeiro,Rio de Janeiro,Rodoviário,"0,00",Não,989.393031,"3940,94","6,31",11/08/2022,20:26


In [ ]:
df_master.columns

Index(['Identificador do processo de viagem', 'Número da Proposta (PCDP)',
       'Situação', 'Viagem Urgente', 'Justificativa Urgência Viagem',
       'Código do órgão superior', 'Nome do órgão superior',
       'Código órgão solicitante', 'Nome órgão solicitante', 'CPF viajante',
       'Nome', 'Cargo', 'Função', 'Descrição Função',
       'Período - Data de início', 'Período - Data de fim', 'Destinos',
       'Motivo', 'Valor diárias', 'Valor passagens', 'Valor devolução',
       'Valor outros gastos', 'Sequência Trecho', 'Origem - Data',
       'Origem - País', 'Origem - UF', 'Origem - Cidade', 'Destino - Data',
       'Destino - País', 'Destino - UF', 'Destino - Cidade',
       'Meio de transporte', 'Número Diárias', 'Missao?', 'Distância (GCD)',
       'Valor da passagem', 'Taxa de serviço', 'Data da emissão/compra',
       'Hora da emissão/compra'],
      dtype='object')

# removendo colunas desnecessarias 

✅ Limpeza de colunas redundantes no df_master concluída para otimizar o GROUPBY (Célula 6).

Colunas restantes:
['Identificador do processo de viagem', 'Número da Proposta (PCDP)', 'Situação', 'Viagem Urgente', 'Justificativa Urgência Viagem', 'Código do órgão superior', 'Nome do órgão superior', 'Código órgão solicitante', 'Nome órgão solicitante', 'CPF viajante', 'Nome', 'Cargo', 'Função', 'Descrição Função', 'Período - Data de início', 'Período - Data de fim', 'Destinos', 'Motivo', 'Valor diárias', 'Valor passagens', 'Valor devolução', 'Valor outros gastos', 'Origem - Cidade', 'Destino - Cidade', 'Meio de transporte', 'Distância (GCD)']


In [ ]:
df_master.columns

Index(['Identificador do processo de viagem', 'Número da Proposta (PCDP)',
       'Situação', 'Viagem Urgente', 'Justificativa Urgência Viagem',
       'Código do órgão superior', 'Nome do órgão superior',
       'Código órgão solicitante', 'Nome órgão solicitante', 'CPF viajante',
       'Nome', 'Cargo', 'Função', 'Descrição Função',
       'Período - Data de início', 'Período - Data de fim', 'Destinos',
       'Motivo', 'Valor diárias', 'Valor passagens', 'Valor devolução',
       'Valor outros gastos', 'Origem - Cidade', 'Destino - Cidade',
       'Meio de transporte', 'Distância (GCD)'],
      dtype='object')

In [ ]:
# Renomeia as colunas que ficaram com sufixo e que são úteis

# Mapeamento para renomear as colunas
colunas_para_renomear = {
    # Renomeia a coluna PCDP mantida
    'Número da Proposta (PCDP)_x': 'Número da Proposta (PCDP)',
}

df_master = df_master.rename(columns=colunas_para_renomear)

print("✅ Colunas com sufixos (_x) renomeadas.")
print("\n--- Amostra das Colunas Finais ---")
print(df_master.columns.tolist())

✅ Colunas com sufixos (_x) renomeadas.

--- Amostra das Colunas Finais ---
['Identificador do processo de viagem', 'Número da Proposta (PCDP)', 'Situação', 'Viagem Urgente', 'Justificativa Urgência Viagem', 'Código do órgão superior', 'Nome do órgão superior', 'Código órgão solicitante', 'Nome órgão solicitante', 'CPF viajante', 'Nome', 'Cargo', 'Função', 'Descrição Função', 'Período - Data de início', 'Período - Data de fim', 'Destinos', 'Motivo', 'Valor diárias', 'Valor passagens', 'Valor devolução', 'Valor outros gastos', 'Origem - Cidade', 'Destino - Cidade', 'Meio de transporte', 'Distância (GCD)']


In [67]:
# --- CÉLULA 5 VECTORIZED  ---
import numpy as np 

# --- CONSTANTES DO MODELO ---
# Fatores de Emissão (EF) em kgCO2eq/km para Classe Econômica
EF_MUITO_CURTA_EVITAVEL = 0.272576785 
EF_CURTA_DISTANCIA = 0.182869354     
EF_LONGA_DISTANCIA = 0.200108215      

# Limites de Distância (em km)
LIMITE_DH = 600   # Muito Curta (Evitável)
LIMITE_SH = 3700  # Curta Distância

# --- 1. DEFINIÇÃO DAS CONDIÇÕES ---

# Condições para np.select
conditions = [
    # 1. Muito Curta (Evitável): Distância <= 600
    df_master['Distância (GCD)'].le(LIMITE_DH),
    # 2. Curta Distância: 600 < Distância <= 3700
    (df_master['Distância (GCD)'].gt(LIMITE_DH)) & (df_master['Distância (GCD)'].le(LIMITE_SH)),
    # 3. Longa Distância: Distância > 3700
    df_master['Distância (GCD)'].gt(LIMITE_SH) 
]

# --- 2. APLICAÇÃO VETORIZADA ---

# Valores correspondentes para a Categoria
category_values = [
    'Muito Curta (Evitável)',
    'Curta Distância',
    'Longa Distância'
]

# Valores correspondentes para o Fator de Emissão (EF)
ef_values = [
    EF_MUITO_CURTA_EVITAVEL,
    EF_CURTA_DISTANCIA,
    EF_LONGA_DISTANCIA
]

# Atribui a Categoria usando np.select
df_master['Categoria Distância'] = np.select(
    conditions, 
    category_values, 
    default='Não Calculada' # Default para NaN ou valores que não se encaixam
)

# Atribui o Fator de Emissão (EF) usando np.select
df_master['EF Aplicado'] = np.select(
    conditions, 
    ef_values, 
    default=np.nan
)

# 3. Cálculo Vetorizado da Emissão: Emissões = Distância * EF
df_master['Emissões (KgCO2eq)'] = df_master['Distância (GCD)'] * df_master['EF Aplicado']

# Opcional: Remove a coluna auxiliar de EF
df_master = df_master.drop(columns=['EF Aplicado'])

print("\n✅ Etapa 5 concluída: Emissões calculadas e categorias aplicadas de forma vetorizada.")
print("\n--- Resultados de Exemplo ---")
print(df_master.head(10)[['Distância (GCD)', 'Categoria Distância', 'Emissões (KgCO2eq)']])


✅ Etapa 5 concluída: Emissões calculadas e categorias aplicadas de forma vetorizada.

--- Resultados de Exemplo ---
   Distância (GCD)     Categoria Distância  Emissões (KgCO2eq)
0      9552.884444         Longa Distância         1911.610654
1      9552.884444         Longa Distância         1911.610654
2       989.393031         Curta Distância          180.929664
3       989.393031         Curta Distância          180.929664
4      9692.698490         Longa Distância         1939.588593
5      9692.698490         Longa Distância         1939.588593
6        10.516381  Muito Curta (Evitável)            2.866521
7        10.516381  Muito Curta (Evitável)            2.866521
8      6833.892697         Longa Distância         1367.518069
9      6833.892697         Longa Distância         1367.518069


In [69]:
# --- CÉLULA 5b: Limpeza de Colunas Antes da Agregação ---

colunas_para_remover = [
    # Detalhes de Trecho (Redundantes após o cálculo de distância)
    'Sequência Trecho', 
    'Origem - Data',
    'Destino - Data',
    'Número Diárias',    # O valor é usado, não a contagem
    'Missao?',
    
    # Detalhes Geográficos (UF e País são redundantes quando temos a Cidade e a Distância)
    'Origem - País',
    'Origem - UF',
    'Destino - País',
    'Destino - UF',

    # Detalhes Financeiros/Temporais (Valores agregados ou muito detalhados)
    'Valor da passagem',   # Custo de um único bilhete (desnecessário quando usamos 'Valor passagens' agregado)
    'Taxa de serviço',
    'Data da emissão/compra',
    'Hora da emissão/compra',
]

# Remove as colunas redundantes
df_master = df_master.drop(columns=colunas_para_remover, errors='ignore')

# Verifica as colunas restantes
print("✅ Limpeza de colunas redundantes no df_master concluída para otimizar o GROUPBY (Célula 6).")
print("\nColunas restantes:")
print(df_master.columns.tolist())

✅ Limpeza de colunas redundantes no df_master concluída para otimizar o GROUPBY (Célula 6).

Colunas restantes:
['Identificador do processo de viagem', 'Número da Proposta (PCDP)_x', 'Situação', 'Viagem Urgente', 'Justificativa Urgência Viagem', 'Código do órgão superior', 'Nome do órgão superior', 'Código órgão solicitante', 'Nome órgão solicitante', 'CPF viajante', 'Nome', 'Cargo', 'Função', 'Descrição Função', 'Período - Data de início', 'Período - Data de fim', 'Destinos', 'Motivo', 'Valor diárias', 'Valor passagens', 'Valor devolução', 'Valor outros gastos', 'Número da Proposta (PCDP)_y', 'Origem - Cidade', 'Destino - Cidade', 'Meio de transporte', 'Distância (GCD)', 'Categoria Distância', 'Emissões (KgCO2eq)']


In [83]:
# --- CÉLULA 5c: IMPUTAÇÃO USANDO MÉDIA (E DESVIO PADRÃO PARA DIAGNÓSTICO) ---
import numpy as np

# --- 0. CONSTANTES DO MODELO (Replicadas de Célula 5 para o Recálculo) ---
# Necessário para aplicar o Fator de Emissão (EF) correto nas distâncias imputadas
EF_MUITO_CURTA_EVITAVEL = 0.272576785 
EF_CURTA_DISTANCIA = 0.182869354     
EF_LONGA_DISTANCIA = 0.200108215      
LIMITE_DH = 600   
LIMITE_SH = 3700  

# --- 1. CÁLCULO DA MÉDIA E DESVIO PADRÃO ---

# Identifica as linhas que precisam de imputação (onde a distância é NaN ou foi preenchida com 0.0)
linhas_para_imputacao = df_master['Distância (GCD)'].isna() | (df_master['Distância (GCD)'] == 0.0)

# Calcula Média e Desvio Padrão APENAS dos valores válidos (> 0)
distancias_validas = df_master['Distância (GCD)'][df_master['Distância (GCD)'] > 0]

if not distancias_validas.empty:
    media_distancia = distancias_validas.mean()
    std_dev_distancia = distancias_validas.std()
    qtd_linhas_imputadas = linhas_para_imputacao.sum()

    # --- 2. IMPUTAÇÃO (Preenchimento com a Média) ---
    # Substituímos as distâncias faltantes pela média das distâncias válidas.
    df_master.loc[linhas_para_imputacao, 'Distância (GCD)'] = media_distancia

    print(f"✅ Imputação concluída. {qtd_linhas_imputadas} linhas preenchidas com a média de {media_distancia:.2f} km.")

    # --- 3. RE-CALCULAR CATEGORIA E EMISSÕES (Apenas para as linhas imputadas) ---

    # Define as condições para o np.select usando a coluna recém-imputada
    conditions = [
        df_master.loc[linhas_para_imputacao, 'Distância (GCD)'] <= LIMITE_DH,
        (df_master.loc[linhas_para_imputacao, 'Distância (GCD)'] > LIMITE_DH) & (df_master.loc[linhas_para_imputacao, 'Distância (GCD)'] <= LIMITE_SH),
        df_master.loc[linhas_para_imputacao, 'Distância (GCD)'] > LIMITE_SH 
    ]
    
    category_values = ['Muito Curta (Evitável)', 'Curta Distância', 'Longa Distância']
    ef_values = [EF_MUITO_CURTA_EVITAVEL, EF_CURTA_DISTANCIA, EF_LONGA_DISTANCIA]

    # Aplica Categoria
    df_master.loc[linhas_para_imputacao, 'Categoria Distância'] = np.select(
        conditions, 
        category_values, 
        default='Imputada' 
    )

    # Aplica EF e Recalcula Emissões
    df_master.loc[linhas_para_imputacao, 'EF Aplicado'] = np.select(conditions, ef_values, default=np.nan)
    df_master.loc[linhas_para_imputacao, 'Emissões (KgCO2eq)'] = df_master.loc[linhas_para_imputacao, 'Distância (GCD)'] * df_master.loc[linhas_para_imputacao, 'EF Aplicado']
    
    # Limpeza
    df_master = df_master.drop(columns=['EF Aplicado'], errors='ignore')

    # --- 4. DIAGNÓSTICO FINAL ---
    print("\n--- Diagnóstico da Distribuição (Dados Válidos) ---")
    print(f"Média de Distância (para imputação): {media_distancia:.2f} km")
    print(f"Desvio Padrão (para diagnóstico): {std_dev_distancia:.2f} km")
    
else:
    print("❌ Aviso: Não há dados válidos de Distância para calcular a Média. Imputação por Média não realizada.")

✅ Imputação concluída. 195267 linhas preenchidas com a média de 960.02 km.

--- Diagnóstico da Distribuição (Dados Válidos) ---
Média de Distância (para imputação): 960.02 km
Desvio Padrão (para diagnóstico): 1511.50 km


# editando aqui

In [84]:
import numpy as np

# --- 0. Conversão de Tipos para Colunas de Custo ---
# A coluna 'Valor da passagem' foi removida na Célula 5b, então removemos ela desta lista.
COLUNAS_CUSTO = ['Valor diárias', 'Valor outros gastos', 'Valor passagens']

for col in COLUNAS_CUSTO:
    # Coerce errors to NaN; se o valor for uma string não numérica, ele se torna NaN.
    # O .str.replace(',', '.', regex=True) é um tratamento comum se houver vírgulas como separador decimal.
    df_master[col] = pd.to_numeric(df_master[col].astype(str).str.replace(',', '.', regex=True), errors='coerce')


# --- 1. AGREGAÇÃO: Nível Trecho para Nível Viagem (Cria DataFrame Agregado Temporário) ---

df_agregado = df_master.groupby('Identificador do processo de viagem').agg(
    # Emissões: Somamos todas as emissões dos trechos de uma mesma viagem
    Total_Emissoes_KgCO2eq=('Emissões (KgCO2eq)', 'sum'),
    
    # Custos: Usamos o primeiro valor das colunas agregadas (pois é replicado por trecho)
    Valor_Passagens=('Valor passagens', 'first'), 
    Valor_Diarias=('Valor diárias', 'first'),
    Valor_Outros=('Valor outros gastos', 'first'),
    
    # Características da Viagem (Pegamos a primeira ocorrência, pois é constante por Viagem ID)
    Viagem_Urgente=('Viagem Urgente', 'first'),
    Justificativa_Urgencia=('Justificativa Urgência Viagem', 'first'),
    
    # Status de Evitável (Para ED1 / viagens curtas)
    Teve_Trecho_Evitavel=('Categoria Distância', lambda x: ('Muito Curta (Evitável)' in x.values))
).reset_index()


# --- 2. CÁLCULO DE MÉTRICAS BASE NO DF AGREGADO ---

# Métrica Custo Total (R$)
df_agregado['Total_Custo_R$'] = df_agregado[['Valor_Passagens', 'Valor_Diarias', 'Valor_Outros']].fillna(0).sum(axis=1)

# Métrica DF1.5: Viagens Urgentes Sem Justificativa
URGENTE_SIM = 'SIM'
SEM_INFO = 'Sem informação'

df_agregado['Viagem_Urgente_Sem_Justificativa'] = (
    (df_agregado['Viagem_Urgente'].astype(str).str.upper().str.strip() == URGENTE_SIM) & 
    (df_agregado['Justificativa_Urgencia'].isna() | 
     (df_agregado['Justificativa_Urgencia'].astype(str).str.upper().str.strip() == SEM_INFO))
)

# Seleciona apenas as novas colunas de interesse para o merge
colunas_para_merge = [
    'Identificador do processo de viagem',
    'Total_Emissoes_KgCO2eq',
    'Total_Custo_R$',
    'Teve_Trecho_Evitavel',
    'Viagem_Urgente_Sem_Justificativa'
]

# --- 3. MERGE DE VOLTA PARA df_master ---

# Une as colunas agregadas de volta ao df_master (que contém todos os detalhes de trecho)
df_master = pd.merge(
    df_master,
    df_agregado[colunas_para_merge],
    on='Identificador do processo de viagem',
    how='left',
    # Adiciona sufixo para evitar conflitos, embora as colunas de merge sejam novas.
    suffixes=('', '_AGREGADO') 
)


print("✅ Agregação para o Nível Viagem concluída.")
print("✅ Colunas de Custo Total, Emissões Totais e DF1.5 adicionadas ao df_master.")
print(f"Colunas agregadas adicionadas: {colunas_para_merge[1:]}")

✅ Agregação para o Nível Viagem concluída.
✅ Colunas de Custo Total, Emissões Totais e DF1.5 adicionadas ao df_master.
Colunas agregadas adicionadas: ['Total_Emissoes_KgCO2eq', 'Total_Custo_R$', 'Teve_Trecho_Evitavel', 'Viagem_Urgente_Sem_Justificativa']


In [85]:
df_master.head()

,Identificador do processo de viagem,Número da Proposta (PCDP)_x,Situação,Viagem Urgente,Justificativa Urgência Viagem,Código do órgão superior,Nome do órgão superior,Código órgão solicitante,Nome órgão solicitante,CPF viajante,...,Categoria Distância,Emissões (KgCO2eq),Total_Emissoes_KgCO2eq,Total_Custo_R$,Teve_Trecho_Evitavel,Viagem_Urgente_Sem_Justificativa,Total_Emissoes_KgCO2eq_AGREGADO,Total_Custo_R$_AGREGADO,Teve_Trecho_Evitavel_AGREGADO,Viagem_Urgente_Sem_Justificativa_AGREGADO
0,0000000000017821923,000001/23-1C,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26352,Fundação Universidade Federal do ABC,***.875.238-**,...,Longa Distância,1911.610654,3823.221308,0.0,False,False,3823.221308,0.0,False,False
1,0000000000017821923,000001/23-1C,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26352,Fundação Universidade Federal do ABC,***.875.238-**,...,Longa Distância,1911.610654,3823.221308,0.0,False,False,3823.221308,0.0,False,False
2,0000000000018159396,000001/23,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26271,Fundação Universidade de Brasília,***.000.000-**,...,Curta Distância,180.929664,4241.036516,7894.5,False,False,4241.036516,7894.5,False,False
3,0000000000018159396,000001/23,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26271,Fundação Universidade de Brasília,***.000.000-**,...,Curta Distância,180.929664,4241.036516,7894.5,False,False,4241.036516,7894.5,False,False
4,0000000000018159396,000001/23,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26271,Fundação Universidade de Brasília,***.000.000-**,...,Longa Distância,1939.588593,4241.036516,7894.5,False,False,4241.036516,7894.5,False,False


In [86]:
df_master.columns

Index(['Identificador do processo de viagem', 'Número da Proposta (PCDP)_x',
       'Situação', 'Viagem Urgente', 'Justificativa Urgência Viagem',
       'Código do órgão superior', 'Nome do órgão superior',
       'Código órgão solicitante', 'Nome órgão solicitante', 'CPF viajante',
       'Nome', 'Cargo', 'Função', 'Descrição Função',
       'Período - Data de início', 'Período - Data de fim', 'Destinos',
       'Motivo', 'Valor diárias', 'Valor passagens', 'Valor devolução',
       'Valor outros gastos', 'Número da Proposta (PCDP)_y', 'Origem - Cidade',
       'Destino - Cidade', 'Meio de transporte', 'Distância (GCD)',
       'Categoria Distância', 'Emissões (KgCO2eq)', 'Total_Emissoes_KgCO2eq',
       'Total_Custo_R$', 'Teve_Trecho_Evitavel',
       'Viagem_Urgente_Sem_Justificativa', 'Total_Emissoes_KgCO2eq_AGREGADO',
       'Total_Custo_R$_AGREGADO', 'Teve_Trecho_Evitavel_AGREGADO',
       'Viagem_Urgente_Sem_Justificativa_AGREGADO'],
      dtype='object')

In [87]:
# --- DIAGNÓSTICO DO CÁLCULO DE DISTÂNCIA E MERGE ---

# 1. Sucesso no cálculo de Distância (Resultado da Célula 3)
# df_trechos_calculados é o DataFrame que contém os trechos únicos e a Distância (GCD)
total_unicos = len(df_trechos_calculados)
sucesso_calculo = df_trechos_calculados['Distância (GCD)'].notna().sum()
perc_sucesso_calculo = (sucesso_calculo / total_unicos) * 100

print("--- DIAGNÓSTICO DO CÁLCULO DE DISTÂNCIA ---")
print(f"Total de Trechos Únicos Tentados: {total_unicos}")
print(f"Trechos com Distância Calculada: {sucesso_calculo}")
print(f"Taxa de Sucesso no Cálculo (API): {perc_sucesso_calculo:.2f}%")

# 2. Sucesso no Merge (Resultado no DataFrame Final)
total_master = len(df_master)
sucesso_merge = df_master['Distância (GCD)'].notna().sum()
perc_sucesso_merge = (sucesso_merge / total_master) * 100

print("\n--- DIAGNÓSTICO NO DATAFRAME FINAL (df_master) ---")
print(f"Total de Linhas em df_master: {total_master}")
print(f"Linhas em df_master com Distância: {sucesso_merge}")
print(f"Percentual de Linhas com Distância: {perc_sucesso_merge:.2f}%")

if sucesso_merge == 0:
    print("\n⚠️ AVISO CRÍTICO: Zero distâncias foram calculadas e/ou merge falhou. A causa é provavelmente o bloqueio da API.")

--- DIAGNÓSTICO DO CÁLCULO DE DISTÂNCIA ---
Total de Trechos Únicos Tentados: 100214
Trechos com Distância Calculada: 100050
Taxa de Sucesso no Cálculo (API): 99.84%

--- DIAGNÓSTICO NO DATAFRAME FINAL (df_master) ---
Total de Linhas em df_master: 1897989
Linhas em df_master com Distância: 1897989
Percentual de Linhas com Distância: 100.00%


In [88]:
# verificar colunas do df_master
df_master.columns

Index(['Identificador do processo de viagem', 'Número da Proposta (PCDP)_x',
       'Situação', 'Viagem Urgente', 'Justificativa Urgência Viagem',
       'Código do órgão superior', 'Nome do órgão superior',
       'Código órgão solicitante', 'Nome órgão solicitante', 'CPF viajante',
       'Nome', 'Cargo', 'Função', 'Descrição Função',
       'Período - Data de início', 'Período - Data de fim', 'Destinos',
       'Motivo', 'Valor diárias', 'Valor passagens', 'Valor devolução',
       'Valor outros gastos', 'Número da Proposta (PCDP)_y', 'Origem - Cidade',
       'Destino - Cidade', 'Meio de transporte', 'Distância (GCD)',
       'Categoria Distância', 'Emissões (KgCO2eq)', 'Total_Emissoes_KgCO2eq',
       'Total_Custo_R$', 'Teve_Trecho_Evitavel',
       'Viagem_Urgente_Sem_Justificativa', 'Total_Emissoes_KgCO2eq_AGREGADO',
       'Total_Custo_R$_AGREGADO', 'Teve_Trecho_Evitavel_AGREGADO',
       'Viagem_Urgente_Sem_Justificativa_AGREGADO'],
      dtype='object')

In [89]:
import os
local = f'dadosViagens/dados_viagens{ano}'
if not os.path.exists(local):
    print(f"📁 Diretório '{local}' não encontrado.")

df_master.to_csv(f'{local}/df_master_viagens_a_servico_{ano}.csv', index=False)
print(f"\n✅ DataFrame Mestre salvo como '{local}/df_master_viagens_a_servico_{ano}.csv'.")


✅ DataFrame Mestre salvo como 'dadosViagens/dados_viagens2023/df_master_viagens_a_servico_2023.csv'.


In [132]:
len(df_master.columns)

45

In [ ]:
#filtra para viagens aereas mantendo todas as colunas

#O filtro deve ser feito no df_master e usar .copy()
aereas = df_master[df_master['Meio de transporte'].astype(str).str.upper().str.strip() == 'AÉREO'].copy()

In [133]:
len(aereas.columns)

45

In [134]:
aereas.head()

,Identificador do processo de viagem,Número da Proposta (PCDP)_x,Situação,Viagem Urgente,Justificativa Urgência Viagem,Código do órgão superior,Nome do órgão superior,Código órgão solicitante,Nome órgão solicitante,CPF viajante,...,Teve_Trecho_Evitavel_AGREGADO,Viagem_Urgente_Sem_Justificativa_AGREGADO,Total_Emissoes_KgCO2eq_AGREGADO,Total_Custo_R$_AGREGADO,Teve_Trecho_Evitavel_AGREGADO,Viagem_Urgente_Sem_Justificativa_AGREGADO,Total_Emissoes_KgCO2eq_AGREGADO,Total_Custo_R$_AGREGADO,Teve_Trecho_Evitavel_AGREGADO,Viagem_Urgente_Sem_Justificativa_AGREGADO
0,0000000000017821923,000001/23-1C,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26352,Fundação Universidade Federal do ABC,***.875.238-**,...,False,False,3823.221308,0.00,False,False,3823.221308,0.00,False,False
1,0000000000017821923,000001/23-1C,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26352,Fundação Universidade Federal do ABC,***.875.238-**,...,False,False,3823.221308,0.00,False,False,3823.221308,0.00,False,False
4,0000000000018159396,000001/23,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26271,Fundação Universidade de Brasília,***.000.000-**,...,False,False,3879.177187,7894.50,False,False,3879.177187,7894.50,False,False
5,0000000000018159396,000001/23,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26271,Fundação Universidade de Brasília,***.000.000-**,...,False,False,3879.177187,7894.50,False,False,3879.177187,7894.50,False,False
8,0000000000018288418,000007/23-1C,Realizada,SIM,Por necessidade do serviço.,52000,Ministério da Defesa,52121,Comando do Exército,***.621.358-**,...,False,False,2735.036138,48095.18,False,False,2735.036138,48095.18,False,False


In [135]:
import os
# salvar aereas 
local = f'dadosViagens/dados_viagens{ano}'
if not os.path.exists(local):
    print(f"📁 Diretório '{local}' não encontrado.")
    
aereas.to_csv(f'{local}/aereas_viagens_a_servico_{ano}.csv', index=False)
print(f"\n✅ DataFrame de Viagens Aéreas salvo como 'aereas")


✅ DataFrame de Viagens Aéreas salvo como 'aereas


In [95]:
aereas.head()

,Identificador do processo de viagem,Número da Proposta (PCDP)_x,Situação,Viagem Urgente,Justificativa Urgência Viagem,Código do órgão superior,Nome do órgão superior,Código órgão solicitante,Nome órgão solicitante,CPF viajante,...,Categoria Distância,Emissões (KgCO2eq),Total_Emissoes_KgCO2eq,Total_Custo_R$,Teve_Trecho_Evitavel,Viagem_Urgente_Sem_Justificativa,Total_Emissoes_KgCO2eq_AGREGADO,Total_Custo_R$_AGREGADO,Teve_Trecho_Evitavel_AGREGADO,Viagem_Urgente_Sem_Justificativa_AGREGADO
0,0000000000017821923,000001/23-1C,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26352,Fundação Universidade Federal do ABC,***.875.238-**,...,Longa Distância,1911.610654,3823.221308,0.00,False,False,3823.221308,0.00,False,False
1,0000000000017821923,000001/23-1C,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26352,Fundação Universidade Federal do ABC,***.875.238-**,...,Longa Distância,1911.610654,3823.221308,0.00,False,False,3823.221308,0.00,False,False
4,0000000000018159396,000001/23,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26271,Fundação Universidade de Brasília,***.000.000-**,...,Longa Distância,1939.588593,4241.036516,7894.50,False,False,4241.036516,7894.50,False,False
5,0000000000018159396,000001/23,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26271,Fundação Universidade de Brasília,***.000.000-**,...,Longa Distância,1939.588593,4241.036516,7894.50,False,False,4241.036516,7894.50,False,False
8,0000000000018288418,000007/23-1C,Realizada,SIM,Por necessidade do serviço.,52000,Ministério da Defesa,52121,Comando do Exército,***.621.358-**,...,Longa Distância,1367.518069,2735.036138,48095.18,False,False,2735.036138,48095.18,False,False


# Metricas 

In [136]:
import numpy as np
import pandas as pd

# --- 0. PREPARAÇÃO E FILTRO (CRÍTICO) ---

# Converte as colunas de custo no df_master primeiro
COLUNAS_CUSTO_ORIGINAIS = ['Valor diárias', 'Valor outros gastos', 'Valor passagens']
for col in COLUNAS_CUSTO_ORIGINAIS:
    # A coluna é convertida para float com tolerância a erros (valores não numéricos viram NaN)
    df_master[col] = pd.to_numeric(df_master[col].astype(str).str.replace(',', '.', regex=True), errors='coerce')

# 🚨 0b. FILTRO DE MEIO DE TRANSPORTE = 'Aéreo' (Cria o DataFrame de Trechos de Análise)
# Usamos 'aereas_trecho' para representar os trechos aéreos
aereas_trecho = df_master[df_master['Meio de transporte'].astype(str).str.upper().str.strip() == 'AÉREO'].copy()
print(f"✅ Filtro de Meio de Transporte ('Aéreo') aplicado. Total de linhas (trechos) para análise: {len(aereas_trecho)}.")


# --- 1. AGREGAÇÃO: Nível Trecho para Nível Viagem ---
# A agregação é feita NO NÍVEL DE TRECHO do DataFrame 'aereas_trecho'.

df_agregado = aereas_trecho.groupby('Identificador do processo de viagem').agg(
    # Emissões: Soma de todos os trechos
    Total_Emissoes_KgCO2eq=('Emissões (KgCO2eq)', 'sum'),
    
    # Custos: Usamos o primeiro valor das colunas agregadas originais
    Valor_Passagens=('Valor passagens', 'first'), 
    Valor_Diarias=('Valor diárias', 'first'),
    Valor_Outros=('Valor outros gastos', 'first'),
    
    # Características da Viagem 
    Viagem_Urgente=('Viagem Urgente', 'first'),
    Justificativa_Urgencia=('Justificativa Urgência Viagem', 'first'),
    
    # Status de Evitável 
    Teve_Trecho_Evitavel=('Categoria Distância', lambda x: ('Muito Curta (Evitável)' in x.values))
).reset_index()


# --- 2. CÁLCULO DE MÉTRICAS BASE NO DF AGREGADO ---

# CRÍTICO: Cálculo de Custo Total
df_agregado['Total_Custo_R$'] = df_agregado[['Valor_Passagens', 'Valor_Diarias', 'Valor_Outros']].fillna(0).sum(axis=1)

# Métrica DF1.5: Viagens Urgentes Sem Justificativa
URGENTE_SIM = 'SIM'
SEM_INFO = 'Sem informação'

df_agregado['Viagem_Urgente_Sem_Justificativa'] = (
    (df_agregado['Viagem_Urgente'].astype(str).str.upper().str.strip() == URGENTE_SIM) & 
    (df_agregado['Justificativa_Urgencia'].isna() | 
     (df_agregado['Justificativa_Urgencia'].astype(str).str.upper().str.strip() == SEM_INFO))
)

# --- 3. FINALIZAÇÃO: REMOVIDO O MERGE DE VOLTA PARA df_master ---

# O resultado final para as próximas células é o DataFrame agregado.
aereas = df_agregado.copy()


print("✅ Agregação, Cálculo e Geração do DataFrame 'aereas' (viagens únicas) concluído.")

✅ Filtro de Meio de Transporte ('Aéreo') aplicado. Total de linhas (trechos) para análise: 528309.
✅ Agregação, Cálculo e Geração do DataFrame 'aereas' (viagens únicas) concluído.


In [127]:
aereas.head()

,Identificador do processo de viagem,Total_Emissoes_KgCO2eq,Valor_Passagens,Valor_Diarias,Valor_Outros,Viagem_Urgente,Justificativa_Urgencia,Teve_Trecho_Evitavel,Total_Custo_R$,Viagem_Urgente_Sem_Justificativa
0,0000000000017821923,3823.221308,0.00,0.0,0.00,NÃO,Sem informação,False,0.00,False
1,0000000000018159396,3879.177187,7894.50,0.0,0.00,NÃO,Sem informação,False,7894.50,False
2,0000000000018288418,2735.036138,7434.13,39565.7,1095.35,SIM,Por necessidade do serviço.,False,48095.18,False
3,0000000000018302983,4215.499282,14903.10,0.0,0.00,NÃO,Sem informação,False,14903.10,False
4,0000000000018306758,4215.499282,14903.10,0.0,0.00,NÃO,Sem informação,False,14903.10,False


In [137]:
# --- CÉLULA 7: GERAÇÃO DE MÉTRICAS E SCORE DF1.5 ---

# Agora 'aereas' é o DataFrame de viagens únicas.
total_viagens = len(aereas)
urgentes_sem_justificativa = aereas['Viagem_Urgente_Sem_Justificativa'].sum()

# 1. GERAÇÃO DA TABELA DE MÉTRICAS BRUTAS
metrics_df = pd.DataFrame({
    'Métrica': [
        'Total Viagens (ED2.3)', 
        'Total Emissões (KgCO2eq, ED1.1)', 
        'Total Custo (R$)', 
        'Viagens Evitáveis (Qtd, <= 600km)', 
        'Viagens Urgentes S/ Just. (DF1.5 Qtd)'
    ],
    'Valor': [
        total_viagens,
        aereas['Total_Emissoes_KgCO2eq'].sum(),
        aereas['Total_Custo_R$'].sum(),
        aereas['Teve_Trecho_Evitavel'].sum(),
        urgentes_sem_justificativa
    ]
})

# 2. CÁLCULO DO INDICADOR DF1.5 SCORE
if total_viagens > 0:
    df1_5_proporcao = urgentes_sem_justificativa / total_viagens
    df1_5_score = 1.0 - df1_5_proporcao
else:
    df1_5_proporcao = 0.0
    df1_5_score = 1.0 

# Tabela de Scores
scores_df = pd.DataFrame({
    'Indicador': ['ED2.3 (Total Viagens)', 'DF1.5 (Proporção)', 'DF1.5 (Score)'],
    'Valor': [total_viagens, df1_5_proporcao, df1_5_score]
})

print("✅ Geração da Tabela de Métricas (metrics_df) e Scores (scores_df) concluída.")
print("\n--- Scores Chave ---")
print(scores_df.to_string(index=False))

✅ Geração da Tabela de Métricas (metrics_df) e Scores (scores_df) concluída.

--- Scores Chave ---
            Indicador         Valor
ED2.3 (Total Viagens) 264071.000000
    DF1.5 (Proporção)      0.000284
        DF1.5 (Score)      0.999716


In [138]:
# --- CÉLULA 7b: DEFINIÇÃO DE CONSTANTES DO MODELO (Hardcoded Baseline) ---
# Estes valores são extraídos da Baseline Oficial do modelo (Baseline 19.2 - 22.2) 
# e são usados como o ponto de comparação fixo para o Score, eliminando a dependência do arquivo Excel.

# Baseline ED1.1 (Emissões Totais em KgCO2eq)
BASELINE_ED1_1 = 285440.32313309913 

# Baseline ED2.1 (Custo Total em R$)
BASELINE_ED2_1 = 1610128.7746285002 

print("✅ Constantes de Baseline ED1.1 e ED2.1 definidas para cálculo independente.")

✅ Constantes de Baseline ED1.1 e ED2.1 definidas para cálculo independente.


In [139]:
# --- CÉLULA 8: CÁLCULO DO SCORE ED1.1 (EMISSÕES) - MODIFICADA ---

# 1. Parâmetros (Não precisamos mais do caminho do Excel)
COLUNA_BASELINE = 'Baseline (19.2 - 22.2)' # Apenas um rótulo de exibição
INDICADOR_ALVO = 'ED1.1'

# 2. Extrair a Baseline ED1.1 (Usando a constante)
baseline_ed1_1 = BASELINE_ED1_1 # Definido na Célula 7b

# Extrai o valor atual de Emissões (da Célula 7)
metrica_atual_emissao = metrics_df[metrics_df['Métrica'] == 'Total Emissões (KgCO2eq, ED1.1)']['Valor'].iloc[0]


# 3. CÁLCULO DO SCORE ED1.1
variacao = (metrica_atual_emissao - baseline_ed1_1) / baseline_ed1_1
score_ed1_1 = 0.0

if metrica_atual_emissao <= baseline_ed1_1:
    score_ed1_1 = 1.0
else:
    score_ed1_1 = max(0, 1.0 - (variacao * 2)) 


# 4. CONSOLIDAÇÃO (Adiciona os resultados ao scores_df)
scores_df.loc[len(scores_df)] = ['ED1.1 (Baseline)', baseline_ed1_1]
scores_df.loc[len(scores_df)] = ['ED1.1 (Variação %)', variacao]
scores_df.loc[len(scores_df)] = ['ED1.1 (Score)', score_ed1_1]


print("✅ Cálculo do Score ED1.1 concluído.")
print(f"\nBaseline ED1.1 Utilizada: {baseline_ed1_1:.2f} KgCO2eq")
print(f"Emissão Atual: {metrica_atual_emissao:.2f} KgCO2eq")
print(f"Variação: {variacao * 100:.2f}%")
print("\n--- Scores Atualizados ---")
print(scores_df.to_string(index=False))

✅ Cálculo do Score ED1.1 concluído.

Baseline ED1.1 Utilizada: 285440.32 KgCO2eq
Emissão Atual: 180565923.42 KgCO2eq
Variação: 63158.73%

--- Scores Atualizados ---
            Indicador         Valor
ED2.3 (Total Viagens) 264071.000000
    DF1.5 (Proporção)      0.000284
        DF1.5 (Score)      0.999716
     ED1.1 (Baseline) 285440.323133
   ED1.1 (Variação %)    631.587300
        ED1.1 (Score)      0.000000


In [140]:
# --- CÉLULA 9: CÁLCULO DO SCORE ED2.1 (CUSTO TOTAL) - MODIFICADA ---

# 1. Parâmetros e Baseline (Usando a constante)
COLUNA_BASELINE = 'Baseline (19.2 - 22.2)' 
INDICADOR_ALVO = 'ED2.1'

# Extrai a Baseline ED2.1 (Usando a constante)
baseline_ed2_1 = BASELINE_ED2_1 # Definido na Célula 7b

# Extrai a Métrica Atual (Custo Total R$)
metrica_atual_custo = metrics_df[metrics_df['Métrica'] == 'Total Custo (R$)']['Valor'].iloc[0]


# 2. CÁLCULO DO SCORE ED2.1
variacao_custo = (metrica_atual_custo - baseline_ed2_1) / baseline_ed2_1
score_ed2_1 = 0.0

if metrica_atual_custo <= baseline_ed2_1:
    score_ed2_1 = 1.0
else:
    # Decaimento do score (usando o mesmo multiplicador de 2)
    score_ed2_1 = max(0, 1.0 - (variacao_custo * 2)) 


# 3. CONSOLIDAÇÃO 
scores_df.loc[len(scores_df)] = ['ED2.1 (Baseline)', baseline_ed2_1]
scores_df.loc[len(scores_df)] = ['ED2.1 (Variação %)', variacao_custo]
scores_df.loc[len(scores_df)] = ['ED2.1 (Score)', score_ed2_1]


print("✅ Cálculo do Score ED2.1 (Custo Total) concluído.")
print(f"\nBaseline ED2.1 Utilizada: {baseline_ed2_1:.2f} R$")
print(f"Custo Atual: {metrica_atual_custo:.2f} R$")
print(f"Variação: {variacao_custo * 100:.2f}%")
print("\n--- Scores Atualizados ---")
print(scores_df.to_string(index=False))

✅ Cálculo do Score ED2.1 (Custo Total) concluído.

Baseline ED2.1 Utilizada: 1610128.77 R$
Custo Atual: 1244578232.36 R$
Variação: 77196.81%

--- Scores Atualizados ---
            Indicador        Valor
ED2.3 (Total Viagens) 2.640710e+05
    DF1.5 (Proporção) 2.840145e-04
        DF1.5 (Score) 9.997160e-01
     ED1.1 (Baseline) 2.854403e+05
   ED1.1 (Variação %) 6.315873e+02
        ED1.1 (Score) 0.000000e+00
     ED2.1 (Baseline) 1.610129e+06
   ED2.1 (Variação %) 7.719681e+02
        ED2.1 (Score) 0.000000e+00
